# Unity Catalog 객체 관리

> **산업 시나리오 — AutoNova Group**
>
> AutoNova Group은 북미, 유럽, 아시아 태평양, 남미 등 4개 지역에 걸쳐 커넥티드 차량을 판매하는 글로벌 자동차 제조업체입니다.
>
> Unity 카탈로그 환경(카탈로그 `trainer_demo`, 스키마 `demo_05`)에는 4개의 테이블이 있습니다.
> - **`vehicles`** — VIN, 제조사, 모델 및 연도가 포함된 차량 재고
> - **`service_records`** — 대리점 서비스 내역
> - **`customers`** — PII를 포함한 등록된 소유자(이메일, 전화)
> - **`telemetry_events`** — 연결된 차량에서 스트리밍되는 실시간 센서 데이터
>
> 데이터 거버넌스 엔지니어로서 귀하는 테이블 정의를 문서화하고, 액세스 정책을 적용하고, 데이터 보존을 관리하고, 데이터 계보를 추적하고, 액세스를 감사하고, 딜러 파트너와 안전한 델타 공유 전략을 설계하게 됩니다.

In [0]:
from scripts.setup import setup, setup_05

spark.sql("USE CATALOG trainer_demo")
setup_05(spark)

## 테이블 정의 생성 및 보존

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.preserve-table-column-definitions.png)

### 🎯 데모: AutoNova 테이블에 설명 및 태그 추가

**강사 참고사항:** 다음 셀을 순서대로 실행하세요.

1. 기존 자동차 테이블 보기  
2. 테이블과 열에 설명 주석 추가  
3. 분류 및 검색을 위한 태그 적용  
4. `DESCRIBE TABLE EXTENDED`을 사용하여 메타데이터 확인

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- Explore the available tables
SHOW TABLES;

In [0]:
%sql
-- Inspect the vehicles table schema
DESCRIBE TABLE EXTENDED vehicles;

In [0]:
%sql
-- 차량 테이블에 대한 설명 코멘트 추가
COMMENT ON TABLE vehicles
IS '## AutoNova 차량 재고
AutoNova Group에서 제조한 모든 등록 차량을 포함합니다.
- **vin**: 차량 식별 번호(VIN) – 차량별로 전 세계적으로 고유함
- **vehicle_type**: 세단, SUV, 트럭, 전기차, 해치백, 쿠페
- **region**: 차량이 등록된 판매 지역';

-- 컬럼별 한글 코멘트 추가
ALTER TABLE vehicles ALTER COLUMN vin     COMMENT '차량 식별 번호 — 17자리 고유 식별자';
ALTER TABLE vehicles ALTER COLUMN vehicle_type COMMENT '차량 유형 분류: 세단, SUV, 트럭, 전기차, 해치백, 쿠페';
ALTER TABLE vehicles ALTER COLUMN region  COMMENT '주요 판매 지역: 북미, 유럽, 아시아 태평양, 남미';

-- 고객 테이블에 대한 한글 코멘트 추가 (PII 포함)
COMMENT ON TABLE customers
IS '## AutoNova 고객 등록부
등록된 차량 소유자 정보. PII 포함 — GDPR 및 지역 데이터 보호법에 따라 처리 필요.
- **email**: 주요 연락 이메일 주소
- **phone**: 국제 형식 전화번호';

ALTER TABLE customers ALTER COLUMN email  COMMENT '주요 이메일 주소 — PII, 삭제 요청 대상';
ALTER TABLE customers ALTER COLUMN phone  COMMENT '국제 형식 전화번호 — PII, 삭제 요청 대상';

-- telemetry_events 테이블 한글 코멘트 추가
COMMENT ON TABLE telemetry_events
IS '## 커넥티드 차량 텔레메트리
AutoNova 커넥티드 차량에서 5분마다 발생하는 실시간 센서 데이터.
- **event_type**: normal | warning | critical | idle
- **engine_temp_c**: 엔진 온도(섭씨)
- **battery_level_pct**: 배터리 충전 상태(0–100)';

In [0]:
%sql
-- Add classification tags to the vehicles table
ALTER TABLE vehicles
SET TAGS ('domain' = 'vehicle_operations', 'data_classification' = 'internal');

-- Tag the customers table — it contains PII
ALTER TABLE customers
SET TAGS ('domain' = 'customer_data', 'data_classification' = 'confidential');

-- Tag individual PII columns
ALTER TABLE customers ALTER COLUMN email
SET TAGS ('pii' = 'email', 'sensitivity' = 'high');

ALTER TABLE customers ALTER COLUMN phone
SET TAGS ('pii' = 'phone', 'sensitivity' = 'high');

-- Tag the VIN as a quasi-identifier
ALTER TABLE vehicles ALTER COLUMN vin
SET TAGS ('sensitivity' = 'medium', 'identifier' = 'vehicle_vin');

-- Tag telemetry for operational use
ALTER TABLE telemetry_events
SET TAGS ('domain' = 'telematics', 'data_classification' = 'internal');

-- Verify tags on the customers table
DESCRIBE TABLE EXTENDED customers;

In [0]:
%sql
-- Use SQL to explore documented assets
SHOW CATALOGS;
SHOW SCHEMAS IN trainer_demo;

-- View tagged columns in the customers table
SHOW COLUMNS IN customers;

-- Check the service_records table definition — add a comment
COMMENT ON TABLE service_records
IS '## 차량 서비스 이력
AutoNova 차량의 대리점 서비스 및 수리 내역. 보증 분석 및 예측 유지보수에 사용됨.
- **service_type**: 수행된 서비스 유형 (오일 교환, 배터리 점검 등)
- **cost_eur**: 서비스 비용(유로)
- **dealer_id**: 인증된 AutoNova 대리점 식별자';

## 태그 및 정책으로 ABAC 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.configure-abac-tags-policies.png)

### 🎯 데모: 관리되는 태그를 사용한 속성 기반 액세스 제어

**강사 참고 사항:** 이 섹션에서는 관리되는 태그를 사용하는 ABAC를 보여줍니다.

1. 관리자는 계정 콘솔 → 카탈로그 → 거버넌스 → 관리되는 태그에서 **관리되는 태그**를 생성합니다.
2. 해당 태그는 민감한 열에 적용됩니다.
3. 열 마스크 및 행 필터 **정책**은 SQL 또는 카탈로그 탐색기를 통해 생성됩니다.

아래 SQL은 개념을 보여줍니다. 정책 생성(`CREATE POLICY`)을 수행하려면 Metastore에서 공개 미리 보기 기능을 활성화해야 합니다.

In [0]:
%sql
-- STEP 1: Apply governed tags to PII columns
-- (Assume governed tag 'pii' with allowed values 'email', 'phone', 'vin' was
--  created in Account Console by an admin)

ALTER TABLE customers ALTER COLUMN email
SET TAGS ('pii' = 'email');

ALTER TABLE customers ALTER COLUMN phone
SET TAGS ('pii' = 'phone');

ALTER TABLE vehicles ALTER COLUMN vin
SET TAGS ('pii' = 'vin');

In [0]:
%sql
-- STEP 2: Create a masking function for email addresses
-- Returns masked email for users who are NOT in the 'data_stewards' group

CREATE OR REPLACE FUNCTION trainer_demo.demo_05.mask_email(email STRING)
RETURNS STRING
DETERMINISTIC
RETURN CASE
    WHEN is_account_group_member('data_stewards') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@***.com')
END;

-- Test the function (when NOT in data_stewards, you see masked output)
SELECT trainer_demo.demo_05.mask_email('luca.rossi99@autonova-demo.com') AS masked_email;

In [0]:
%sql
    
-- STEP 3: Create a column mask policy using the UDF and governed tag
-- This policy automatically applies to any column tagged pii=email in the schema
-- NOTE: Requires ABAC Public Preview to be enabled on your metastore

CREATE POLICY mask_customer_email
ON SCHEMA trainer_demo.demo_05
COLUMN MASK trainer_demo.demo_05.mask_email
TO `account users`
EXCEPT `data_stewards`
FOR TABLES
MATCH COLUMNS
    hasTagValue('pii', 'email') AS email_col
ON COLUMN email_col;

In [0]:
# 정책 적용 테스트: 마스킹, 행 필터, 태그, 보존 등
# 1. 고객 이메일 마스킹 정책 테스트
display(spark.sql("""
SELECT customer_id, email, trainer_demo.demo_05.mask_email(email) AS masked_email
FROM customers
LIMIT 10
"""))

# 2. 지역별 행 필터 정책 테스트
display(spark.sql("""
SELECT customer_id, region
FROM customers
GROUP BY customer_id, region
ORDER BY region
"""))

# 3. 태그 적용 확인 (DESCRIBE EXTENDED)
display(spark.sql("""
DESCRIBE TABLE EXTENDED customers
"""))

# 4. 보존 정책 확인 (TBLPROPERTIES)
display(spark.sql("""
SHOW TBLPROPERTIES telemetry_events
"""))

# 5. 계보 정책 확인 (system.access.table_lineage)
display(spark.sql("""
SELECT source_table_full_name, target_table_full_name, event_time
FROM system.access.table_lineage
WHERE event_date > CURRENT_DATE() - INTERVAL 7 DAYS
  AND (source_table_full_name LIKE 'trainer_demo.demo_05.%'
       OR target_table_full_name LIKE 'trainer_demo.demo_05.%')
ORDER BY event_time DESC
LIMIT 10
"""))

In [0]:
# 열마스크 정책이 적용된 상태 확인
# 현재 사용자(jaiee.shin@ikss.kr)는 data_stewards 그룹에 속해 마스킹이 해제됨
# j@kisedkoss.onmicrosoft.com 처럼 data_stewards에 속하지 않은 사용자가 보는 결과를 시뮬레이션:

# 1. 현재 사용자(data_steward)가 보는 결과 — 마스킹 없음
print("=== data_stewards 그룹 사용자(현재 사용자) 결과 ===")
display(spark.sql("""
SELECT customer_id, email
FROM customers
LIMIT 10
"""))

# 2. data_stewards에 속하지 않은 사용자(j@kisedkoss.onmicrosoft.com)가 보는 결과 시뮬레이션
print("\n=== 비 data_stewards 사용자(j@kisedkoss.onmicrosoft.com) 시뮬레이션 ===")
display(spark.sql("""
SELECT customer_id, email,
       CONCAT(LEFT(email, 2), '***@***.com') AS email_as_seen_by_non_steward
FROM customers
LIMIT 10
"""))

In [0]:
%sql
-- STEP 4: Row filter — regional compliance example
-- Only analysts in the 'eu_analysts' group should see European customer records
-- Others see only non-European customers

CREATE OR REPLACE FUNCTION trainer_demo.demo_05.filter_eu_region(region STRING)
RETURNS BOOLEAN
RETURN CASE
    WHEN is_account_group_member('eu_analysts') THEN TRUE
    ELSE region != 'Europe'
END;

-- Apply as a row filter policy on the customers table
ALTER TABLE customers SET ROW FILTER trainer_demo.demo_05.filter_eu_region ON (region);

-- Verify: non-EU analysts should not see European rows
SELECT region, COUNT(*) AS customer_count
FROM customers
GROUP BY region
ORDER BY region;

## 데이터 보존 정책 적용

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/govern-unity-catalog-objects/media/5-understand-data-retention-settings.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.apply-data-retention-policies.png)

### 🎯 데모: 차량 원격 측정 데이터 보존 정책
**강사 참고 사항:**  
AutoNova는 운영 목적으로 90일간 원격 측정 데이터를 보관합니다.  
GDPR 등 규정에 따라 고객의 개인 데이터는 요청 시 영구 삭제할 수 있어야 합니다.  
이 섹션에서는 다음을 보여줍니다:  
1. Delta Lake 보존 속성 조회 및 설정  
2. `VACUUM`으로 오래된 데이터 파일 제거  
3. GDPR 삭제 권리 요청 처리  
4. 예측 최적화로 자동 유지 관리 활성화

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- View current retention settings on the telemetry table
SHOW TBLPROPERTIES telemetry_events;

In [0]:
%sql
-- Set retention on telemetry_events to 90 days
-- AutoNova only needs 90 days of telemetry history for SLA reporting
ALTER TABLE telemetry_events
SET TBLPROPERTIES (
    'delta.logRetentionDuration'        = 'interval 90 days',
    'delta.deletedFileRetentionDuration' = 'interval 90 days'
);

-- Set longer retention on service_records for warranty compliance (7 years)
ALTER TABLE service_records
SET TBLPROPERTIES (
    'delta.logRetentionDuration'        = 'interval 2555 days',
    'delta.deletedFileRetentionDuration' = 'interval 2555 days'
);

SHOW TBLPROPERTIES telemetry_events;

In [0]:
%sql
-- VACUUM: remove data files older than the retention window
-- (DRY RUN first — shows what would be deleted without actually deleting)
VACUUM telemetry_events DRY RUN;

-- Run actual VACUUM after confirming the dry run output
-- VACUUM telemetry_events;

In [0]:
%sql
-- GDPR right-to-erasure: delete a customer and their telemetry
-- Simulates a deletion request for customer_id = 42

-- Step 1: Delete the customer record
DELETE FROM customers WHERE customer_id = 42;

-- Step 2: Delete their telemetry (linked via vehicle_id)
DELETE FROM telemetry_events
WHERE vehicle_id IN (
    SELECT vehicle_id FROM customers WHERE customer_id = 42
);

-- Step 3: Verify deletion
SELECT COUNT(*) AS remaining_records FROM customers WHERE customer_id = 42;

-- Step 4: Run VACUUM to permanently remove data files (irreversible!)
-- VACUUM customers;
-- VACUUM telemetry_events;

In [0]:
%sql
-- Enable predictive optimization on demo_05 schema
-- This automates VACUUM and OPTIMIZE across all managed tables
ALTER SCHEMA trainer_demo.demo_05
ENABLE PREDICTIVE OPTIMIZATION;

-- Check if predictive optimization is active on the telemetry table
DESCRIBE TABLE EXTENDED telemetry_events;

## 데이터 계보 설정 및 관리

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.manage-data-lineage-tracking.png)

### 🎯 데모: Unity 카탈로그의 데이터 계보

**강사 참고 사항:**  
Unity Catalog는 Unity Catalog 지원 컴퓨팅에서 실행되는 쿼리의 계보를 자동으로 캡처합니다.  
이 섹션에서는 다음을 보여줍니다:  
1. 계보 관계를 생성하기 위한 파생 테이블 만들기  
2. `DESCRIBE HISTORY`로 테이블 기록 보기  
3. 시스템 계보 테이블 쿼리(`system.access.table_lineage`)  

**팁:** 셀을 실행하는 동안 `vehicles_with_service`의 카탈로그 탐색기에 **계보** 탭을 표시하세요.

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- Create a summary view that derives from vehicles, customers, and service_records
-- This will appear as a downstream node in the lineage graph
CREATE OR REPLACE TABLE vehicles_with_service AS
SELECT
    v.vehicle_id,
    v.vin,
    v.make,
    v.model,
    v.year,
    v.vehicle_type,
    c.region,
    COUNT(s.service_id)     AS total_services,
    SUM(s.cost_eur)         AS total_service_cost_eur,
    MAX(s.service_date)     AS last_service_date
FROM vehicles v
LEFT JOIN service_records s ON v.vehicle_id = s.vehicle_id
LEFT JOIN customers c       ON s.customer_id = c.customer_id
GROUP BY v.vehicle_id, v.vin, v.make, v.model, v.year, v.vehicle_type, c.region;

SELECT * FROM vehicles_with_service LIMIT 5;

In [0]:
%sql
-- View the change history of the vehicles table
DESCRIBE HISTORY vehicles;

-- Show only the latest 5 versions
DESCRIBE HISTORY vehicles LIMIT 5;

In [0]:
%sql
-- Query Unity Catalog system lineage tables
-- Shows which tables were read/written in the last 7 days

SELECT
    source_table_full_name,
    target_table_full_name,
    COUNT(DISTINCT event_id)  AS event_count,
    MAX(event_time)           AS last_event_time
FROM system.access.table_lineage
WHERE event_date > CURRENT_DATE() - INTERVAL 7 DAYS
  AND (
        source_table_full_name LIKE 'trainer_demo.demo_05.%'
     OR target_table_full_name LIKE 'trainer_demo.demo_05.%'
  )
GROUP BY source_table_full_name, target_table_full_name
ORDER BY last_event_time DESC;

In [0]:
%sql
-- View and change table ownership
-- (Requires metastore admin or current owner role)
DESCRIBE TABLE EXTENDED vehicles_with_service;

-- Transfer ownership example (replace with actual user/group)
-- ALTER TABLE vehicles_with_service OWNER TO `data-engineering-team@autonova.com`;

## 감사 로깅 구성

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.configure-audit-logging.png)

### 🎯 데모: system.access.audit를 사용한 감사 로깅
**강사 참고 사항:**  
Azure Databricks는 모든 감사 이벤트를 `system.access.audit`에 저장합니다.  
이 섹션에서는 AutoNova의 규정 준수 팀이 실행할 일반적인 거버넌스 쿼리를 보여줍니다:

1. 민감한 고객 데이터에 접근한 사람 찾기  
2. 메타스토어 전체의 권한 변경 추적  
3. 의심스러운 실패한 액세스 시도 식별  
4. 노트북 명령 실행 모니터링(자세한 로깅)

In [0]:
%sql
-- Find who queried the customers table (contains PII) in the last 7 days
SELECT
    user_identity.email                     AS user_email,
    action_name                             AS access_type,
    event_time                              AS access_time,
    source_ip_address
FROM system.access.audit
WHERE
    request_params.full_name_arg = 'trainer_demo.demo_05.customers'
    AND action_name IN ('getTable', 'createTable', 'deleteTable', 'updateTable')
    AND event_date > CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY event_time DESC
LIMIT 50;

In [0]:
%sql
-- Track all permission changes across the trainer_demo catalog
-- Useful for auditing who granted/revoked access to vehicle and customer data
SELECT
    event_time,
    user_identity.email                     AS changed_by,
    request_params.securable_type           AS object_type,
    request_params.securable_full_name      AS object_name,
    request_params.changes                  AS permission_changes
FROM system.access.audit
WHERE
    service_name  = 'unityCatalog'
    AND action_name = 'updatePermissions'
    AND request_params.securable_full_name LIKE 'trainer_demo%'
ORDER BY event_time DESC
LIMIT 30;

In [0]:
%sql
-- Identify failed access attempts — potential security incident indicator
SELECT
    event_time,
    user_identity.email                     AS user_email,
    source_ip_address,
    action_name,
    response.error_message
FROM system.access.audit
WHERE
    service_name          = 'accounts'
    AND response.status_code != 200
    AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC
LIMIT 30;

In [0]:
%sql
-- Monitor job and cluster activity related to AutoNova data pipelines
SELECT
    event_time,
    user_identity.email                     AS triggered_by,
    action_name,
    request_params.job_id                   AS job_id,
    response.status_code
FROM system.access.audit
WHERE
    service_name = 'jobs'
    AND event_date > CURRENT_DATE() - INTERVAL 7 DAYS
ORDER BY event_time DESC
LIMIT 30;

## 안전한 델타 공유 전략 설계

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/govern-unity-catalog-objects.secure-delta-sharing-strategy.png)

### 🎯 데모: 딜러 파트너와의 안전한 델타 공유

**강사 참고 사항:**  
AutoNova는 Delta Sharing을 통해 공인 대리점과 집계된 차량 서비스 데이터를 공유합니다.  
딜러는 차량에 대한 서비스 요약에 액세스해야 하지만 원시 고객 PII는 액세스할 수 없습니다.  

이 섹션에서는 다음을 보여줍니다:  
1. 공유 생성 및 집계 보기 추가  
2. Databricks-Databricks 수신자 만들기  
3. 수신자에게 공유에 대한 액세스 권한 부여  
4. 공유 활동 모니터링

In [0]:
%sql
USE CATALOG trainer_demo;
USE SCHEMA demo_05;

-- Create a dealer-safe view: no PII, only vehicle service summaries
CREATE OR REPLACE VIEW dealer_service_summary AS
SELECT
    s.dealer_id,
    v.make,
    v.model,
    v.year,
    v.vehicle_type,
    s.service_type,
    AVG(s.cost_eur)          AS avg_cost_eur,
    COUNT(s.service_id)      AS service_count,
    MAX(s.service_date)      AS latest_service_date
FROM service_records s
JOIN vehicles v ON s.vehicle_id = v.vehicle_id
GROUP BY s.dealer_id, v.make, v.model, v.year, v.vehicle_type, s.service_type;

SELECT * FROM dealer_service_summary LIMIT 10;

In [0]:
%sql
    
-- Create a Delta Share for authorised dealer partners
-- (Using a unique name to avoid ownership conflicts with pre-existing shares)
CREATE SHARE IF NOT EXISTS autonova_dealer_share_demo05
COMMENT 'Aggregated vehicle service data for authorised AutoNova dealerships — no PII included';

-- Add the dealer-safe view to the share
ALTER SHARE autonova_dealer_share_demo05
ADD VIEW trainer_demo.demo_05.dealer_service_summary
COMMENT 'Service cost and frequency summary by dealer, vehicle type, and service type';

-- Optionally share the vehicles table (no PII columns)
ALTER SHARE autonova_dealer_share_demo05
ADD TABLE trainer_demo.demo_05.vehicles;

-- Inspect what is in the share
SHOW ALL IN SHARE autonova_dealer_share_demo05;

In [0]:
%sql
    
-- Create a Databricks-to-Databricks recipient for a dealer network partner
-- The recipient provides their metastore sharing identifier:
--   SELECT CURRENT_METASTORE();  (run in their workspace)

CREATE RECIPIENT IF NOT EXISTS contoso_dealer_network
USING ID 'azure:koreacentral:061c5534-7b7e-4e95-bb4d-af503246055e'  -- replace with actual UUID
COMMENT 'Contoso Dealer Network — authorised AutoNova partner';

-- Grant the recipient access to the share
GRANT SELECT ON SHARE autonova_dealer_share_demo05 TO RECIPIENT contoso_dealer_network;

-- Verify grants
SHOW GRANT ON SHARE autonova_dealer_share_demo05;

In [0]:
%sql
-- Monitor share access: who accessed the share and when
SELECT
    event_time,
    user_identity.email                     AS recipient_user,
    action_name,
    request_params.share_name
FROM system.access.audit
WHERE
    service_name  = 'deltaSharingServer'
    AND event_date > CURRENT_DATE() - INTERVAL 30 DAYS
ORDER BY event_time DESC
LIMIT 30;

In [0]:
%sql
-- Clean-up / management commands (reference only — do not run in demo)

-- Remove a table from a share
-- ALTER SHARE autonova_dealer_share REMOVE TABLE trainer_demo.demo_05.vehicles;

-- Revoke a recipient's access
-- REVOKE SELECT ON SHARE autonova_dealer_share FROM RECIPIENT contoso_dealer_network;

-- Drop a share (removes all assets and access grants)
-- DROP SHARE IF EXISTS autonova_dealer_share;

-- List all shares visible to you
SHOW SHARES;